# LC #217 — Contains Duplicate
**Category:** HashSet | **Difficulty:** Easy | **Day 1**

---

<div style="padding:15px;border-left:8px solid #667eea;background:#f0f0ff;border-radius:4px;">
<strong>The Problem:</strong> Given an integer array <code>nums</code>, return <code>true</code> if
any value appears at least twice. Return <code>false</code> if every element is distinct.
</div>

**Examples:**
```
Input: [1, 2, 3, 1]  → true   (1 appears twice)
Input: [1, 2, 3, 4]  → false  (all distinct)
Input: [1, 1, 1, 3]  → true
```

**Core Insight:** You need O(1) membership testing as you scan. A HashSet is exactly that — add each element, and if it's already there, you found a duplicate. Exit immediately (early termination).

## Mental Models

**1. The Seen Set**
Maintain a set of everything encountered so far. For each new element: "Have I seen you before?" If yes — duplicate found. If no — add to the set, continue. The set is your memory.

**2. Early Termination**
You don't need to process the entire array. The moment you find a duplicate, return `True`. This matters in practice — duplicates near the front of large arrays mean very fast execution.

**3. Set vs HashMap**
A set is a hashmap where you only care about keys, not values. Use a set when you only need membership testing (is X in the collection?). Use a dict when you also need to store data alongside each key (Two Sum stores the index).

In [ ]:
# Approach 1 — Sort, then check adjacent
# O(n log n) time, O(1) extra space (modifies input)

def containsDuplicate_sort(nums):
    nums.sort()  # adjacent duplicates will be neighbors
    for i in range(1, len(nums)):
        if nums[i] == nums[i - 1]:
            return True
    return False

# Approach 2 — Compare length with set length (Pythonic one-liner)
# O(n) time but builds the full set before checking
def containsDuplicate_oneliner(nums):
    return len(nums) != len(set(nums))

# Test
print(containsDuplicate_sort([1, 2, 3, 1]))   # True
print(containsDuplicate_sort([1, 2, 3, 4]))   # False

## Why HashSet Beats Both

**Sort approach:** O(n log n) and **mutates** the input array. You often can't destroy the caller's data. Also slower.

**One-liner `set(nums)`:** Correct and Pythonic, but builds the **entire** set before any comparison. If the first two elements are duplicates, it still processes all n elements. No early exit.

**HashSet with early exit:** Processes elements one at a time. Stops the moment a duplicate is found. For arrays like `[1, 1, 2, 3, ..., 999999]`, this returns after 2 iterations instead of n.

In production data pipelines, you often deduplicate event streams. Early exit means you can flag bad data faster.

In [ ]:
# Optimal — O(n) time, O(n) space
# HashSet with early termination.

def containsDuplicate(nums: list[int]) -> bool:
    seen = set()
    for num in nums:
        if num in seen:
            return True   # found duplicate — exit immediately
        seen.add(num)
    return False

# One-liner alternative (no early exit, but clean):
# return len(nums) != len(set(nums))

# Test
print(containsDuplicate([1, 2, 3, 1]))    # True
print(containsDuplicate([1, 2, 3, 4]))    # False
print(containsDuplicate([1, 1, 1, 3, 3])) # True

## Complexity Analysis

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Brute Force (nested loops) | O(n²) | O(1) | Compare every pair |
| Sort + adjacent check | O(n log n) | O(1) | Mutates input |
| `len(nums) != len(set(nums))` | O(n) | O(n) | No early exit |
| **HashSet with early exit** | **O(n)** | **O(n)** | **Stops on first dup** |

**Space note:** In the worst case (no duplicates), the set holds all n elements. O(n) space.

**Practical note:** For very large arrays with early duplicates, the expected runtime is much better than O(n). The O(n) is the worst case (no duplicate or duplicate at the very end).

## Interview Q&A

**Q: When would you prefer the sort approach over the hashset?**
A: When memory is extremely constrained — O(1) extra space vs O(n). In embedded systems or when the array is nearly sorted already. Sorting in-place (`.sort()` in Python) uses O(log n) stack space for Timsort, essentially O(1) extra.

**Q: Why does `len(nums) != len(set(nums))` work?**
A: A set only keeps unique elements. If any duplicates exist, `set(nums)` has fewer elements than `nums`. If lengths are equal, all elements are unique.

**Q: What's the difference between a Python `set` and `frozenset`?**
A: `set` is mutable (you can add/remove). `frozenset` is immutable and hashable — you can use it as a dict key or store it inside another set.

**Q: Could this problem appear in a data engineering context?**
A: Constantly. Deduplication is a core ETL operation — ensuring metric IDs are unique, detecting double-counted events, validating primary keys before loading to a data warehouse. The HashSet pattern is fundamental.

## The Citi Angle

**ETL deduplication:** At Citi, telemetry agents sometimes sent duplicate metric records (network retries, agent restarts). Before aggregating, we needed to detect and drop duplicates.

**Direct analogy:**
```python
# Checking for duplicate server_id in a batch before inserting
seen_ids = set()
for record in batch:
    if record['server_id'] in seen_ids:
        log_duplicate(record)
        continue
    seen_ids.add(record['server_id'])
    insert(record)
```

**Interview tie-in:** "Deduplication before aggregation was a daily concern at Citi — 6,000 endpoints with retry logic meant we saw duplicate metric records. The HashSet pattern — add and check — is exactly what prevents double-counting in capacity utilization reports."

## Summary

| | Value |
|--|--|
| **Pattern** | HashSet — membership testing with early exit |
| **Time** | O(n) worst case, much better average |
| **Space** | O(n) |
| **Key line** | `if num in seen: return True` |
| **Say in interview** | "HashSet for O(1) membership testing. Early exit on first duplicate." |

**When to choose which:**
- Need early exit + no mutation → HashSet loop (this solution)
- Memory constrained → Sort approach
- Idiomatic Python, no early exit needed → `len(nums) != len(set(nums))`